In [0]:
%python
df = spark.read.format("delta")\
    .option("header", True)\
    .option("inferschema", True)\
    .load("abfss://bronze@mynetflixstorage3.dfs.core.windows.net/netflix_MovieorSeries_titles")
display(df)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df= df.fillna({'duration_minutes':0, 'duration_seasons':0})
display(df)

In [0]:
df=df.withColumn('duration_minutes', df['duration_minutes'].cast(IntegerType()))
df.display()

In [0]:
df=df.withColumn('duration_seasons', df['duration_seasons'].cast(IntegerType()))
df.display()


In [0]:
df=df.withColumn('Short_title',split(col('title'),':')[0])
df.display()

In [0]:
df=df.withColumn('rating',split(col('rating'),'-')[0])
df.display()

In [0]:
df=df.withColumn('type_flag', when(col('type')=='Movie','less duration').when(col('type')=='TV Show','long duration').otherwise('0'))
df.display()

In [0]:
from pyspark.sql import Window

In [0]:
df=df.withColumn('duration_ranking',dense_rank().over(Window.orderBy(col('duration_minutes').desc())))
df.display()

In [0]:
df_vis=df.groupBy('type').agg(count("*").alias("total_count"))
display(df_vis)

Databricks visualization. Run in Databricks to view.

In [0]:
df.write.format('delta')\
    .mode('overwrite')\
    .option("path","abfss://silver@mynetflixstorage3.dfs.core.windows.net/netflix_titles")\
    .save()